# 05b Contrast sweep

Reruns the noise-free matrix imaging of `05_matrix_imaging` for one or two scenarios at velocity drops of 1, 2.5, 5
and 10 % inside the BDT, imaging with P waves and with S waves, and finds the smallest drop the best design can
still detect. The Vs drop is tied to the Vp drop by `VS_RATIO` (Vs drop / Vp drop), which depends on what causes
the low-velocity zone; `docs/02_vs_contrast_notes.md` gives the values and where they come from:

| Cause | Vs drop / Vp drop | Vp/Vs |
|---|---|---|
| fluid-filled cracks (dense brine or supercritical fluid), or melt | ~2.2 | up |
| temperature alone | ~1.4 | slightly up |
| vapour-like fluid in cracks (unlikely at 4–6 km) | ~0.6 | down |

**Read this before trusting the answer.** In 05 a column counts as imaged when its peak is `SNR_MIN` times the
image's *own* median. With no noise the Born image scales almost exactly linearly with the drop, so that relative
test gives the same coverage at every drop. Each image is therefore scored twice:

- **relative**: 05's own score, unchanged (flat across drops);
- **absolute**: the same score with the detection floor fixed at the background of that phase's `DROP_REF` image.
  This stands in for a noise level until `06_noise` exists, and the answer depends on it. P and S each get their
  own floor, so comparing them assumes the noise is the same fraction of the P and S signal.

Inputs: everything 05 reads, including the S travel-time tables from 03 (`PHASES = ("p", "s")`).
Outputs: `data/results_05b_contrast.csv`, images in `data/mi_images_{p,s}_drop*/`

In [ ]:
# --- Imports ---
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# --- Settings ---
DROPS = [0.01, 0.025, 0.05, 0.10]           # fractional drop in the velocity being imaged (Vp for P, Vs for S)
SWEEP_PHASES = ["p", "s"]
SWEEP_SCENARIOS = ["pos00_dome", "pos09_steeper"]   # one centred, one off-centre and narrow
BEST = None                                 # design to test; None = highest mean coverage in results_05_mi.csv
DROP_REF = 0.05                             # its image background sets the fixed (absolute) detection floor
DETECT_COV = 0.5                            # detected = at least this fraction of the BDT top imaged within TOP_TOL
DETECT_CREST = 1.0                          # ... and the BDT high picked within this many km
MECHANISMS = {"fluid-filled cracks / melt": 2.2, "temperature alone": 1.4, "vapour-like fluid": 0.6}  # Vs/Vp drop
OUT = "../data/results_05b_contrast.csv"

In [ ]:
# --- 05's code: settings, inputs and functions (everything before its run cell), once per phase ---
src = "".join("".join(c["source"]) + "\n" for c in json.load(open("05_matrix_imaging.ipynb"))["cells"]
              if c["cell_type"] == "code")
cells = src.split("# %%")
RUN = next(i for i, c in enumerate(cells) if "# --- Run:" in c)
setup, run_cell = "# %%".join(cells[:RUN]), cells[RUN]
assert 'PHASE = "p"' in setup


def load_05(phase):
    ns = {}
    exec(setup.replace('PHASE = "p"', f"PHASE = {phase!r}", 1), ns)
    return ns


NS = {ph: load_05(ph) for ph in SWEEP_PHASES}
DATA = NS[SWEEP_PHASES[0]]["DATA"]
if BEST is None:
    BEST = pd.read_csv(f"{DATA}/results_05_mi.csv").groupby("design").coverage.mean().idxmax()
print("best design:", BEST)


def tag(phase, drop):
    return f"{phase}_drop{drop:g}"

In [ ]:
# --- Sweep: run 05's run cell once per phase and drop, for the best design only ---
for ph, ns in NS.items():
    for drop in DROPS:
        ns.update(DROP=drop, SCENARIOS=SWEEP_SCENARIOS, DESIGNS=[BEST],
                  RESULTS=f"{DATA}/results_05_mi_{tag(ph, drop)}.csv", IMAGE_DIR=f"{DATA}/mi_images_{tag(ph, drop)}")
        print(f"--- {ph.upper()}, drop {drop:.1%}")
        exec(run_cell, ns)

In [ ]:
# --- Score each image against 05's relative floor and against a fixed floor from the DROP_REF image ---
key = BEST.replace(" ", "_")


def load(ph, drop, name):
    return np.load(f"{DATA}/mi_images_{tag(ph, drop)}/{name}.npz")[key]


def score_floor(ns, img, name, floor):
    """05's score, but a column is imaged only if its peak is SNR_MIN x floor (floor fixed across drops)."""
    centre = np.array(ns["SC"]["scenarios"][name]["centre"])
    true = ns["sc_npz"][f"top_{name}"][np.ix_(ns["mx"][ns["ii"]], ns["my"][ns["ij"]])]
    est = ns["pick_top"](img, ns["SNR_MIN"] * floor)
    XX, YY = np.meshgrid(ns["IX"], ns["IY"], indexing="ij")
    target = (np.hypot(XX - centre[0], YY - centre[1]) <= ns["TARGET_R"]) & np.isfinite(true)
    good = target & (np.abs(est - true) <= ns["TOP_TOL"])
    strong = ns["pick_top"](img, max(ns["CREST_FRAC"] * img.max(), ns["SNR_MIN"] * floor))
    crest_err = (float(np.hypot(*(ns["crest"](strong, XX, YY) - ns["crest"](true, XX, YY))))
                 if np.isfinite(strong).any() else np.nan)
    return good.sum() / max(target.sum(), 1), crest_err


rows = []
for ph, ns in NS.items():
    for name in SWEEP_SCENARIOS:
        floor = float(np.median(load(ph, DROP_REF, name)))
        for drop in DROPS:
            img = load(ph, drop, name)
            rel = pd.read_csv(f"{DATA}/results_05_mi_{tag(ph, drop)}.csv").query(
                "scenario == @name and design == @BEST").iloc[0]
            cov_abs, crest_abs = score_floor(ns, img, name, floor)
            rows.append(dict(phase=ph, scenario=name, design=BEST, drop=drop,
                             peak_over_floor=float(img.max() / floor),
                             coverage_rel=rel.coverage, crest_err_rel=rel.crest_err_km,
                             coverage_abs=cov_abs, crest_err_abs=crest_abs,
                             detected=bool(cov_abs >= DETECT_COV and crest_abs <= DETECT_CREST)))
res = pd.DataFrame(rows)
res.to_csv(OUT, index=False)
res.round(3)

In [ ]:
# --- Smallest detected drop per phase and scenario (grid value, and the coverage crossing interpolated in log drop) ---
def min_drop(g):
    g = g.sort_values("drop")
    ok = g.detected.to_numpy()
    if not ok.any():
        return np.nan, np.nan
    k = int(np.argmax(ok))
    if k == 0:
        return g["drop"].iloc[0], np.nan                 # detected at the smallest drop tried: limit is lower
    (d0, d1), (c0, c1) = g["drop"].iloc[k - 1:k + 1], g.coverage_abs.iloc[k - 1:k + 1]
    f = np.clip((DETECT_COV - c0) / (c1 - c0), 0, 1) if c1 > c0 else 1.0
    return d1, float(np.exp(np.log(d0) + f * (np.log(d1) - np.log(d0))))


lim = pd.DataFrame([dict(phase=ph, scenario=name, **dict(zip(("min_drop_grid", "min_drop_interp"), min_drop(g))))
                   for (ph, name), g in res.groupby(["phase", "scenario"])])
print(lim.to_string(index=False, float_format=lambda v: f"{v:.1%}"))
print("relative (05) coverage varies by at most "
      f"{res.groupby(['phase', 'scenario']).coverage_rel.agg(np.ptp).max():.3f} across drops")

In [ ]:
# --- Smallest detectable Vp drop for each cause of the LVZ: P needs Vp drop >= its limit, S needs Vs drop >= its
#     limit, i.e. Vp drop >= S limit / VS_RATIO ---
best = lim.assign(d=lim.min_drop_interp.fillna(lim.min_drop_grid)).pivot(index="scenario", columns="phase", values="d")
out = []
for mech, ratio in MECHANISMS.items():
    for name, r in best.iterrows():
        vp_p = r.get("p", np.nan)
        vp_s = r.get("s", np.nan) / ratio
        out.append({"cause": mech, "Vs/Vp drop": ratio, "scenario": name, "min Vp drop (P image)": vp_p,
                    "min Vp drop (S image)": vp_s, "better": ("neither" if np.isnan(vp_p) and np.isnan(vp_s)
                               else "S" if np.isnan(vp_p) or vp_s < vp_p else "P")})
pd.DataFrame(out).round(4)

In [ ]:
# --- Coverage against drop ---
with plt.style.context("default"):
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
    for ax, col, lab in zip(axs, ["coverage_rel", "coverage_abs"],
                            ["Relative floor (05 as is)", f"Fixed floor (background at {DROP_REF:.0%})"]):
        for (ph, name), g in res.groupby(["phase", "scenario"]):
            ax.plot(g["drop"] * 100, g[col], "o-" if ph == "p" else "s--", label=f"{ph.upper()}: {name}")
        ax.axhline(DETECT_COV, color="grey", ls="--", lw=0.8)
        ax.set(xscale="log", xticks=[1, 2.5, 5, 10], xticklabels=["1", "2.5", "5", "10"],
               xlabel="Drop in imaged velocity (%)", title=lab, ylim=(0, 1.02))
    axs[0].set_ylabel(f"Coverage of BDT top ({BEST})")
    axs[1].legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()